In [ ]:
# ============================================================
# 📘 Discounted Cash Flow (DCF) & Reverse DCF Calculator
# ============================================================
# HOW TO USE:
# 1️⃣ Fill in the inputs below under "User Settings".
# 2️⃣ Leave ONE variable as None:
#       - price = None  → compute the fair value per share
#       - g = None      → compute the implied FCF growth rate (reverse DCF)
#       - g_n = None    → compute the implied terminal growth rate
# 3️⃣ Click ▶ Run All (or Runtime → Run all)
#    If Colab warns that this notebook isn’t from Google, click "Run Anyway".
# 4️⃣ The results appear at the end of the cell — scroll down to view them.
# ============================================================

"""
Created on Thu Oct  9 06:45:43 2025

"""

# ============================================================
# 📘 Discounted Cash Flow (DCF) & Reverse DCF Calculator
# ============================================================
# HOW TO USE:
# 1️⃣ Fill in the inputs below under "User Settings".
# 2️⃣ Leave ONE variable as None:
#       - price = None  → compute the fair value per share
#       - g = None      → compute the implied FCF growth rate (reverse DCF)
#       - g_n = None    → compute the implied terminal growth rate
# 3️⃣ Click ▶ Run All (or Runtime → Run all)
# The script will automatically detect which variable to solve for.
# 4️⃣ If fair value is computed, a sensitivity chart will show how
#     value changes with growth rate or discount rate.
# ============================================================

# --- User Settings ------------------------------------------------------------
FCF_0   = 72_000          # current free cash flow in millions (e.g., from 10-K)
n       = 10               # explicit forecast horizon in years
r       = 0.085            # discount rate (e.g., 9%)
g       = None            # None if you want to compute implied FCF growth
g_n     = 0.05            # terminal growth rate (None to solve for this)
shares  = 7_433_000_000   # number of shares outstanding
price   = 528.57          # share price (None to compute fair value)
#NOTE: use same currency for FCF_0 and price if both are used as inputs
# -----------------------------------------------------------------------------
#---DO NOT CHANGE THE CODE BELOW-----------------------------------------------

import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt

# --- Core DCF calculation -----------------------------------------------------
def dcf_value(FCF_0, g, r, g_n, n):
    """Present value of forecast FCFs plus terminal value."""
    fcf_forecast = [FCF_0 * ((1 + g) / (1 + r))**t for t in range(1, n + 1)]
    terminal_val = (FCF_0 * (1 + g)**n * (1 + g_n) / (r - g_n)) / (1 + r)**n
    return sum(fcf_forecast) + terminal_val  # total firm value in millions

# --- Solver logic -------------------------------------------------------------
if g is None:
    # Solve for implied growth rate (reverse DCF)
    market_value = price * shares / 1e6
    func = lambda g_guess: dcf_value(FCF_0, g_guess, r, g_n, n) - market_value
    g = fsolve(func, 0.10)[0]
    #g = brentq(func, 0.0, 0.3)  # search for growth rate between 0% and 30%
    result_type = "Implied annual FCF growth rate"
    result_value = g
elif price is None:
    # Compute fair value per share
    total_value = dcf_value(FCF_0, g, r, g_n, n)
    fair_value = (total_value * 1e6) / shares
    result_type = "Estimated fair value per share"
    result_value = fair_value
elif g_n is None:
    # Solve for implied terminal growth rate
    market_value = price * shares / 1e6
    func = lambda g_n_guess: dcf_value(FCF_0, g, r, g_n_guess, n) - market_value
    g_n = fsolve(func, 0.02)[0]
    result_type = "Implied terminal growth rate"
    result_value = g_n
else:
    raise ValueError("Please set one of (price, g, g_n) to None to compute it.")

# 💡 NOTE:
# The results will appear near the *end* of the cell output below.
# Scroll down if you don’t immediately see the summary table.
# --- Output -------------------------------------------------------------------
print("============================================================")
print(" Discounted Cash Flow (DCF) Analysis ")
print("============================================================")
print(f"FCF_0 (latest)           : {FCF_0:,.0f} M")
print(f"Horizon (n)              : {n} years")
print(f"Discount rate (r)        : {r*100:.1f} %")

# Conditional prints for optional inputs
if g is not None:
    print(f"Growth rate (g)          : {g*100:.2f} %")
if g_n is not None:
    print(f"Terminal growth (g_n)    : {g_n*100:.2f} %")
if price is not None:
    print(f"Market price             : {price:,.2f}")

print(f"Number of shares         : {shares:,.0f}")


print("------------------------------------------------------------")
if 'growth' in result_type.lower():
    print(f"Result.  {result_type}: {result_value*100:.2f} %")
else:
    print(f"Result.  {result_type}: {result_value:,.2f}")
print("------------------------------------------------------------")

total_val = dcf_value(FCF_0, g, r, g_n, n)
print(f"Total DCF value (millions): {total_val:,.0f}")
print(f"Per-share value (computed): {(total_val*1e6/shares):.2f}")
print("============================================================")

# --- Sensitivity analysis -----------------------------------------------------
if price is None:  # Only when we computed a fair value
    # Sensitivity to growth rate
    g_values = np.linspace(max(0, g - 0.05), g + 0.05, 50)
    fair_values = [(dcf_value(FCF_0, gi, r, g_n, n) * 1e6 / shares) for gi in g_values]

    # Sensitivity to discount rate
    r_values = np.linspace(r - 0.02, r + 0.02, 50)
    fair_values_r = [(dcf_value(FCF_0, g, ri, g_n, n) * 1e6 / shares) for ri in r_values]

    #fig, ax = plt.subplots(2, 1, figsize=(4,12))
    fig, ax = plt.subplots(2, 1)
    
    ax[0].plot(g_values * 100, fair_values, color='blue')
    ax[0].axvline(g * 100, color='gray', linestyle='--')
    ax[0].set_title("Fair Value vs. FCF Growth (g)")
    ax[0].set_xlabel("FCF Growth Rate (%)")
    ax[0].set_ylabel("Fair Value per Share ($)")
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(r_values * 100, fair_values_r, color='green')
    ax[1].axvline(r * 100, color='gray', linestyle='--')
    ax[1].set_title("Fair Value vs. Discount Rate (r)")
    ax[1].set_xlabel("Discount Rate (%)")
    ax[1].set_ylabel("Fair Value per Share ($)")
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()